<a href="https://colab.research.google.com/github/dsc-courses/cosmos-ml-cluster-2026/blob/main/labs/lab10/lab10_nn_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks

In this lab, you are you going to train a multi-layer perceptron (MLP) neural network on an object classifcation dataset called CIFAR10, which consists of thousands of 32 x 32 RGB images and each image has an object you are trying to classify. Please take a quick look [here](https://www.cs.toronto.edu/~kriz/cifar.html) to learn more about the dataset before you get started. Your task will be to design both neural network models, train them, and compare their classifciation performances. You will ultimately choose which model is better and justify your answer.

In your presentations, you can discuss which MLP architecture you used (number of layers, number of neurons per layer, the max training iterations, the training time, its training and testing accuracy, its confusion matrix)

Make sure you experiment with different neural network designs as you make your models so that you get the highest accuracy possible on your testing data. Let's see which group is able to get the highest accuracy!

# Load and vizualize data

In [ ]:
# import libraries
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt


The below cell is actually a more standardized way of downloading the CIFAR-10 datset. However it actually takes **1.5hr** just to download it .. :) To save our time, I already downloaded the dataset to our class server, unzipped it - so you can download it from there.

### Standardized way: DO NOT RUN THE CELL BELOW

In [ ]:
# import data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# reshape y_train, y_val, and y_test to appropriate shape
y_train = np.array(y_train).reshape((y_train.shape[0], ))
y_test = np.array(y_test).reshape((y_test.shape[0], ))

# print the shape of all the training data. What do all the numbers in the shapes
# represent? Put your answer as a comment below



### Instead, RUN THIS

In [ ]:
import os

# Corrected GitHub repository URL
github_repo_url = "https://github.com/dsc-courses/cosmos-ml-cluster-2026"
repo_name = github_repo_url.split('/')[-1]

# Path to the specific subfolder within the repository
subfolder_relative_path = "labs/lab10/data/cifar-10-batches-py"

# Local directory where we want to store these files
local_target_folder = os.path.join(os.getcwd(), 'data', 'cifar-10-batches-py')

# Check if the folder already exists and has expected files to avoid re-downloading
# Checking for 'data_batch_1' as a common indicator file
if os.path.exists(os.path.join(local_target_folder, 'data_batch_1')):
    print(f"Folder '{local_target_folder}' already exists and contains data batches. Skipping download.")
else:
    print(f"Downloading files from {github_repo_url}/{subfolder_relative_path}...")

    # Clone the repository into /content
    cloned_repo_dir = os.path.join(os.getcwd(), repo_name)
    if not os.path.exists(cloned_repo_dir):
        print(f"Cloning repository: {github_repo_url} into {cloned_repo_dir}...")
        get_ipython().system(f'git clone {github_repo_url} {cloned_repo_dir}')
    else:
        print(f"Repository already cloned at {cloned_repo_dir}. Skipping clone.")

    # Source path of the target subfolder within the cloned repository
    source_path_in_cloned_repo = os.path.join(cloned_repo_dir, subfolder_relative_path)

    # Ensure the local target directory exists
    os.makedirs(local_target_folder, exist_ok=True)

    # Copy contents from the cloned repo's subfolder to the local target directory
    print(f"Copying contents from {source_path_in_cloned_repo} to {local_target_folder}...")
    # Use shell copy command for robust folder content copy
    get_ipython().system(f'cp -r {source_path_in_cloned_repo}/* {local_target_folder}/')

    print(f"Files successfully downloaded and copied to {local_target_folder}.")

    # Clean up the cloned repository (optional, but good practice in Colab)
    print(f"Cleaning up cloned repository: {cloned_repo_dir}...")
    get_ipython().system(f'rm -rf {cloned_repo_dir}')
    print("Cleanup complete.")

# Verify contents
print(f"\nContents of {local_target_folder}:")
if os.path.exists(local_target_folder):
    print(os.listdir(local_target_folder))
else:
    print("Target directory does not exist after operation.")

print("\nNote: The CIFAR-10 dataset is typically downloaded automatically by `tf.keras.datasets.cifar10.load_data()`. This code directly downloads the specific files from the GitHub repository as requested, which might be useful for custom data loading or verification purposes.")

In [ ]:
import pickle

def unpickle(file_path):
    with open(file_path, 'rb') as fo:
        dict_data = pickle.load(fo, encoding='bytes')
    return dict_data

# The local_target_folder variable is available from the previous cell's execution
# local_target_folder = '/content/data/cifar-10-batches-py'

x_train_list = []
y_train_list = []

# Load training batches (data_batch_1 to data_batch_5)
for i in range(1, 6):
    batch_file_path = os.path.join(local_target_folder, f'data_batch_{i}')
    data_dict = unpickle(batch_file_path)
    x_train_list.append(data_dict[b'data'])
    y_train_list.extend(data_dict[b'labels'])

# Concatenate and convert to numpy arrays
x_train_raw = np.concatenate(x_train_list)
y_train_raw = np.array(y_train_list)

# Load test batch
test_batch_file_path = os.path.join(local_target_folder, 'test_batch')
test_dict = unpickle(test_batch_file_path)
x_test_raw = test_dict[b'data']
y_test_raw = np.array(test_dict[b'labels'])

# Reshape image data from (N, 3072) to (N, 32, 32, 3)
# The data is stored in row-major order, with 3 channels (R, G, B) interleaved.
# Each channel's 1024 values are contiguous (first 1024 for R, then 1024 for G, then 1024 for B).
# So reshape to (N, 3, 32, 32) and then transpose to (N, 32, 32, 3)
x_train = x_train_raw.reshape((len(x_train_raw), 3, 32, 32)).transpose(0, 2, 3, 1)
x_test = x_test_raw.reshape((len(x_test_raw), 3, 32, 32)).transpose(0, 2, 3, 1)

# Normalize pixel values to be between 0 and 1
x_train = x_train / 255.0
x_test = x_test / 255.0

# Assign y_train and y_test from their raw versions
y_train = y_train_raw
y_test = y_test_raw


In [ ]:
# train and test sets are defined. Check out the shapes.
print (x_train.shape)
print (y_train.shape)
print (x_test.shape)
print (y_test.shape)

### Check out some sample images from `x_train`

In [ ]:
# Display some sample images from the training set (run as-is)
# Note the images will look very low-quality because they are 32 x 32 images
# that we are enlarging
plt.figure(figsize=(10, 10))
for i in range(9):
  plt.subplot(3, 3, i + 1)
  plt.imshow(x_train[i, :, :, :])
  plt.title(f"Class: {y_train[i]}")
  plt.axis('off')
plt.show()

# MLP Training
Write code to prepare and train your MLP below.